# Exercise 3 - indices

## Detailed Description of Methods

This notebook utilizes several advanced remote sensing and data processing methods:

1. **Hyperspectral Subset Loading**: Given the massive size of hyperspectral cubes, we use `spectral.io.envi` to read only a specific `subregion`. This allows us to perform analysis on a representative 1km x 1km area without overloading system memory.
2. **Spatial Coordinate Transformation**: Airborne data is captured in a local projected coordinate system (e.g., Poland CS2000). We use `rasterio.warp.transform` to convert these coordinates into WGS84 (latitude/longitude), which is required for global satellite data discovery via STAC.
3. **SpatioTemporal Asset Catalog (STAC) Querying**: Instead of manually searching for data, we programmatically query the Microsoft Planetary Computer. We filter for Sentinel-2 Level-2A (Bottom of Atmosphere) products based on geographic intersection, a specific time window, and a strict cloud cover threshold (<10%).
4. **Geospatial Window Alignment**: To ensure the airborne and satellite data are comparable, we use `rasterio.windows.from_bounds` to clip the Sentinel-2 image to the exact geographic footprint of our airborne subset.
5. **Spectral Index Extraction**: We extract specific narrowband reflectance values for Chl-a (705nm), DOC (560nm), and Turbidity (700nm). For Sentinel-2, we use the closest broadband equivalents (B05, B03, and B11 as a turbidity proxy).

## Imports

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform
from rasterio.windows import from_bounds
import spectral.io.envi as envi
from pystac_client import Client
import planetary_computer
from datetime import datetime

# Set acquisition date
ACQUISITION_DATE = '2025-06-17'

# Relative path to airborne data
BASE_DIR = Path().cwd() / "data" / "you-shall-not-pass" / "Obrazy lotnicze"
hdr_path = BASE_DIR / '221000_Odra_HS_Blok_A_008_VS_join_atm.hdr'
bsq_path = BASE_DIR / '221000_Odra_HS_Blok_A_008_VS_join_atm.bsq'

print(f"Target Date: {ACQUISITION_DATE}")
print(f"Airborne Data Path: {bsq_path}")

## Load Airborne data

In [ ]:
if not hdr_path.exists():
    raise FileNotFoundError(f"Cannot find airborne data at {hdr_path}")

img = envi.open(hdr_path)
meta = img.metadata
wavelengths = np.array([float(x) for x in meta['wavelength']])

# Read a subset (1000x1000) from the center to keep memory usage low
cy, cx = img.nrows // 2, img.ncols // 2
cube = img.read_subregion((cy-500, cy+500), (cx-500, cx+500))

# Extract geographic bounds of the subset using the .bsq file for rasterio
with rasterio.open(bsq_path) as src:
    air_win = rasterio.windows.Window(cx-500, cy-500, 1000, 1000)
    air_left, air_bottom, air_right, air_top = src.window_bounds(air_win)
    lons, lats = transform(src.crs, 'EPSG:4326', [air_left, air_right], [air_bottom, air_top])
    subset_bbox = [lons[0], lats[0], lons[1], lats[1]]

print(f"Loaded subset shape: {cube.shape}")

## Show false-color 

In [ ]:
r_idx = np.argmin(np.abs(wavelengths - 670))
g_idx = np.argmin(np.abs(wavelengths - 560))
b_idx = np.argmin(np.abs(wavelengths - 490))

rgb_air = cube[:, :, [r_idx, g_idx, b_idx]]
rgb_air = (rgb_air - np.percentile(rgb_air, 2)) / (np.percentile(rgb_air, 98) - np.percentile(rgb_air, 2))
rgb_air = np.clip(rgb_air, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(rgb_air)
plt.title("Airborne False Color RGB")
plt.axis('off')
plt.show()

## Water Quality Indices: Technical Notes

The following indices are used to estimate water constituents from spectral reflectance:

1. **Chlorophyll-a (Chl-a)**: This is the primary pigment found in phytoplankton. In water quality monitoring, it serves as a proxy for algal biomass and eutrophication. We use the **Red Edge/Red ratio (705nm / 665nm)**. This ratio is effective because 665nm corresponds to a chlorophyll absorption peak, while 705nm (the "red edge") is highly reflective for vegetation/algae, making the ratio sensitive to concentration changes.

2. **Dissolved Organic Carbon (DOC)**: Often measured via CDOM (Chromophoric Dissolved Organic Matter), this represents organic material that stains the water. We use the **Green/Red ratio (560nm / 665nm)**. DOC strongly absorbs blue and green light; by comparing the green reflectance to the red, we can estimate the concentration of these dissolved components.

3. **Turbidity**: This measures water clarity and the presence of suspended particulate matter (sediment, silt, or organic debris). 
    - In the **airborne** data, we use reflectance at **700nm**, where the signal is dominated by backscatter from suspended solids.
    - In **Sentinel-2**, we use the **B11 (SWIR)** band as a proxy. Since water strongly absorbs SWIR radiation, any reflectance in this band typically indicates the presence of suspended matter or surface floating material.

## Water Quality Indices (Airborne)

In [ ]:
def safe_ratio(n, d, eps=1e-6):
    return n / (d + eps)

idx_705 = np.argmin(np.abs(wavelengths - 705))
idx_665 = np.argmin(np.abs(wavelengths - 665))
idx_560 = np.argmin(np.abs(wavelengths - 560))
idx_700 = np.argmin(np.abs(wavelengths - 700))

chla_air = safe_ratio(cube[:, :, idx_705], cube[:, :, idx_665])
doc_air = safe_ratio(cube[:, :, idx_560], cube[:, :, idx_665])
turb_air = cube[:, :, idx_700]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (data, name) in enumerate(zip([chla_air, doc_air, turb_air], ['Chl-a', 'DOC', 'Turbidity'])):
    im = axes[i].imshow(data, cmap='viridis')
    axes[i].set_title(f"Airborne {name}")
    plt.colorbar(im, ax=axes[i])
plt.show()

In [ ]:
import certifi
import os
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

## Download Sentinel-2 Data

In [ ]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

# Strict cloud cover and exact alignment search
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=subset_bbox,
    datetime=ACQUISITION_DATE,
    query={"eo:cloud_cover": {"lt": 10}}
)

items = list(search.items())
if not items:
    print("Exact date cloudy or missing. Searching nearest 15 days, sorted by cloud cover...")
    search = catalog.search(
        collections=["sentinel-2-l2a"], 
        bbox=subset_bbox, 
        datetime="2025-06-10/2025-06-25", 
        query={"eo:cloud_cover": {"lt": 10}}
    )
    items = sorted(list(search.items()), key=lambda x: x.properties["eo:cloud_cover"])

if not items:
    raise ValueError("No suitable cloud-free Sentinel-2 imagery found.")

best_item = items[0]
print(f"Selected Sentinel-2 Scene: {best_item.id} ({best_item.datetime})")
print(f"Cloud Cover: {best_item.properties['eo:cloud_cover']:.2f}%")

assets = best_item.assets
bands_s2 = {
    'B03': assets['B03'].href, # Green
    'B04': assets['B04'].href, # Red
    'B05': assets['B05'].href, # Red Edge 1
    'B11': assets['B11'].href  # SWIR 1
}

## Water Quality Indices (Sentinel-2)

In [ ]:
data_s2 = {}
with rasterio.open(bands_s2['B04']) as src:
    # Match window to airborne subset coordinates
    s2_win = from_bounds(subset_bbox[0], subset_bbox[1], subset_bbox[2], subset_bbox[3], src.transform)
    data_s2['B04'] = src.read(1, window=s2_win).astype(float) / 10000.0

with rasterio.open(bands_s2['B03']) as src:
    data_s2['B03'] = src.read(1, window=s2_win).astype(float) / 10000.0

with rasterio.open(bands_s2['B05']) as src:
    data_s2['B05'] = src.read(1, window=s2_win, out_shape=data_s2['B04'].shape, resampling=rasterio.enums.Resampling.bilinear).astype(float) / 10000.0

with rasterio.open(bands_s2['B11']) as src:
    data_s2['B11'] = src.read(1, window=s2_win, out_shape=data_s2['B04'].shape, resampling=rasterio.enums.Resampling.bilinear).astype(float) / 10000.0

chla_s2 = safe_ratio(data_s2['B05'], data_s2['B04'])
doc_s2 = safe_ratio(data_s2['B03'], data_s2['B04'])
turb_s2 = data_s2['B11']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(chla_s2, cmap='viridis', vmin=0, vmax=2)
axes[0].set_title("Sentinel-2 Chl-a Proxy")
axes[1].imshow(doc_s2, cmap='viridis', vmin=0, vmax=2)
axes[1].set_title("Sentinel-2 DOC Proxy")
axes[2].imshow(turb_s2, cmap='viridis')
axes[2].set_title("Sentinel-2 Turbidity Proxy")
plt.show()

## Compare index values

In [ ]:
print(f"Airborne Chl-a: {np.nanmean(chla_air):.4f}")
print(f"Sentinel-2 Chl-a: {np.nanmean(chla_s2):.4f}")
print(f"Relative Difference (Chl-a): {abs(np.nanmean(chla_air)-np.nanmean(chla_s2))/np.nanmean(chla_air)*100:.2f}%")

print(f"\nAirborne DOC: {np.nanmean(doc_air):.4f}")
print(f"Sentinel-2 DOC: {np.nanmean(doc_s2):.4f}")
print(f"Relative Difference (DOC): {abs(np.nanmean(doc_air)-np.nanmean(doc_s2))/np.nanmean(doc_air)*100:.2f}%")

print(f"\nAirborne Turbidity: {np.nanmean(turb_air):.4f}")
print(f"Sentinel-2 Turbidity: {np.nanmean(turb_s2):.4f}")
print(f"Relative Difference (Turbidity): {abs(np.nanmean(turb_air)-np.nanmean(turb_s2))/np.nanmean(turb_air)*100:.2f}%")

## Visual comparison

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 18))

# Chl-a
im0 = axes[0, 0].imshow(chla_air, cmap='viridis', vmin=0, vmax=2)
axes[0, 0].set_title("Airborne Chl-a")
plt.colorbar(im0, ax=axes[0, 0])
im1 = axes[0, 1].imshow(chla_s2, cmap='viridis', vmin=0, vmax=2)
axes[0, 1].set_title("Sentinel-2 Chl-a")
plt.colorbar(im1, ax=axes[0, 1])

# DOC
im2 = axes[1, 0].imshow(doc_air, cmap='viridis', vmin=0, vmax=2)
axes[1, 0].set_title("Airborne DOC")
plt.colorbar(im2, ax=axes[1, 0])
im3 = axes[1, 1].imshow(doc_s2, cmap='viridis', vmin=0, vmax=2)
axes[1, 1].set_title("Sentinel-2 DOC")
plt.colorbar(im3, ax=axes[1, 1])

# Turbidity
im4 = axes[2, 0].imshow(turb_air, cmap='viridis')
axes[2, 0].set_title("Airborne Turbidity")
plt.colorbar(im4, ax=axes[2, 0])
im5 = axes[2, 1].imshow(turb_s2, cmap='viridis')
axes[2, 1].set_title("Sentinel-2 Turbidity")
plt.colorbar(im5, ax=axes[2, 1])

plt.tight_layout()
plt.show()